In [ ]:
import pymc as pm
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import arviz as az
import xarray as xr
az.style.use("arviz-variat")
%matplotlib inline

In [ ]:
def visualise_kernel(cov, *, x0=0, x1=2, k=200, ax=None, size=1, method="plot", jitter=1e-6, seed=42, **kwargs):
    # cov = cov + pm.gp.cov.WhiteNoise(1e-6)
    X = np.linspace(x0, x1, k)[:, None]
    K = cov(X).eval()
    K[np.diag_indices_from(K)] += jitter * np.trace(K) / len(K)
    if ax is None:
        ax = plt.gca()
    f = pm.draw(pm.MvNormal.dist(mu=np.zeros(len(K)), cov=K, size=size), random_seed=seed).T
    getattr(ax, method)(X, f, **kwargs)

In [ ]:
lengthscales = [100]
plt.figure(figsize=(14, 4))
for i, l in enumerate(lengthscales):
    eta = 2.0
    cov = eta**2 * pm.gp.cov.ExpQuad(1, l)
    # Добавляем белый шум для численной стабильности
    visualise_kernel(cov, x0=0, x1=1000, color=f"C{i}", label=fr"$ls={l}$")


plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");
plt.legend();

K * sd**2

In [ ]:
amplitudes = [0.25, 0.5, 10]
plt.figure(figsize=(14, 4))
for i, am in enumerate(amplitudes):
    cov = am**2 * pm.gp.cov.ExpQuad(1, 0.2)
    # Добавляем белый шум для численной стабильности
    visualise_kernel(cov, color=f"C{i}", label=fr"$\sigma={am}$")
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");
plt.legend()

In [ ]:
white_noise = [0.25, 0.5, 10]

for i, noi in enumerate(white_noise):
    cov = pm.gp.cov.ExpQuad(1, 0.2) + pm.gp.cov.WhiteNoise(noi)
    visualise_kernel(cov, color=f"C{i}", label=fr"$\sigma_*={noi}$", method="scatter")
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");
plt.legend()

In [ ]:
periods = [0.125, 0.25, 0.5]
plt.figure(figsize=(14, 4))
for i, period in enumerate(periods):
    var = 3
    w = 0.8
    cov = pm.gp.cov.ExpQuad(1, ls=0.5) * (w) + pm.gp.cov.Periodic(1, period=period, ls=0.5)*(1-w)
    # Добавляем белый шум для численной стабильности
    visualise_kernel(cov, color=f"C{i}", label=fr"$period={period}$")
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");
plt.legend()

In [ ]:
periods = [1, 0.25, 1/12, 1/12/4]
lengthscales = [1, 0.25, 1/12*2, 1/12/4*2]
weights = [0.7, 0.2, 0.08, 0.02]
plt.figure(figsize=(14, 4))
cov = sum(
    [pm.gp.cov.Periodic(1, period=p, ls=ls) * w for p, ls, w in zip(periods, lengthscales, weights)]
)
# Добавляем белый шум для численной стабильности
visualise_kernel(cov, x1=1, k=1000, size=1)
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");
plt.xlim(0, 4/12)

In [ ]:
plt.figure(figsize=(14, 4))
cov = pm.gp.cov.Linear(1, c=0) * 10 + pm.gp.cov.Periodic(1, period=0.25, ls=0.5)
# Добавляем белый шум для численной стабильности
visualise_kernel(cov, size=3)
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");

In [ ]:
plt.figure(figsize=(14, 4))
cov = pm.gp.cov.ExpQuad(1, ls=0.1) * pm.gp.cov.Periodic(1, period=0.25, ls=0.3)
# Добавляем белый шум для численной стабильности
visualise_kernel(cov, size=3)
plt.title("Samples from the GP prior")
plt.ylabel("y")
plt.xlabel("X");

In [ ]:
J = 8
y = np.array([28, 8, -3, 7, -1, 1, 18, 12])
# снижаем дисперсию наблюдений, чтобы графики были более читаемыми
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18]) / 2

In [ ]:
with pm.Model(coords=dict(J=range(J))) as schools_centered:
    mu = pm.Normal("mu", mu=0, sigma=5)
    tau = pm.HalfCauchy("tau", beta=5)
    theta = pm.Normal("theta", mu=mu, sigma=tau, dims="J") # средний балл по школе
    obs = pm.Normal("obs", mu=theta, sigma=sigma, observed=y)
    trace0 = pm.sample()

In [ ]:
with pm.Model(coords=dict(J=range(J))) as schools_noncentered:
    mu = pm.Normal("mu", mu=0, sigma=5)
    tau = pm.HalfCauchy("tau", beta=5)
    theta_tilde = pm.Normal("theta_t", mu=0, sigma=1, dims="J")
    theta = pm.Deterministic("theta", mu + tau * theta_tilde, dims="J")
    obs = pm.Normal("obs", mu=theta, sigma=sigma, observed=y)
    trace1 = pm.sample(target_accept=0.90)

In [ ]:
az.plot_forest(trace1, var_names=["theta"])

In [ ]:
school_coords = np.array(
    [[  0.,  10.],
     [  8., -11.],
     [  0.,  -4.],
     [ -1.,  -3.],
     [ 16.,   4.],
     [ 17.,  19.],
     [  7., 5.],
     [ 5.,  -7.]])

gridxy = np.mgrid[
    school_coords[:, 0].min()-1:school_coords[:, 0].max()+1:20j,
    school_coords[:, 1].min()-1:school_coords[:, 1].max()+1:20j
]
grid = gridxy.reshape(2, -1).T

In [ ]:
trace1.posterior.theta.mean(["chain", "draw"]).values

In [ ]:
theta_mean = trace1.posterior.theta.mean(["chain", "draw"]).values
s = trace1.posterior.theta.std(["chain", "draw"]).values
plt.scatter(*school_coords.T, c=(theta_mean-min(theta_mean))/(max(theta_mean)-min(theta_mean)), s=s*30, edgecolors="black")
[plt.text(x, y+1, str(i)) for i, (x, y) in enumerate(school_coords)];
plt.xlabel("x")
plt.ylabel("y")
plt.title("Schools coordinates")

In [ ]:
plt.scatter(*school_coords.T, c=(theta_mean-min(theta_mean))/(max(theta_mean)-min(theta_mean)))
[plt.text(x, y+1, str(i)) for i, (x, y) in enumerate(school_coords)];
plt.xlabel("x")
plt.ylabel("y")
plt.title("Schools coordinates")

In [ ]:
X = np.random.randn(100, 2)
Y = np.sin(X[:, 0]*5)*4 + (X[:, 1]) ** 2 + np.random.randn(100) *0.01

In [ ]:
plt.scatter(*X.T, c=(Y-min(Y))/(max(Y)-min(Y)))
plt.title("Schools coordinates")

In [ ]:
with pm.Model() as gp_visual:
    ls = pm.Gamma("ls", mu=1, sigma=0.3)
    magnitude = pm.Exponential("magnitude", 1)

    cov = pm.gp.cov.ExpQuad(2, ls=ls) * magnitude
    
    white_noise = pm.Exponential("white_noise", 50)
    mean = pm.Normal("mean", 0, 1)
    
    
    gp = pm.gp.Marginal(mean_func=pm.gp.mean.Constant(mean), cov_func=cov)
    
    
    gp.marginal_likelihood("lik", X, Y, sigma=white_noise)

In [ ]:
with gp_visual:
    prior = pm.sample_prior_predictive()

In [ ]:
prior.prior_predictive.lik.mean()

In [ ]:
lik_mean = prior.prior_predictive.lik.mean(["draw", "chain"])
prior.prior_predictive.lik.std(["draw", "chain"])
plt.scatter(*X.T, c=(lik_mean-min(lik_mean))/(max(lik_mean)-min(lik_mean)))
[plt.text(x, y, str(i)) for i, (x, y) in enumerate(X)];
plt.xlabel("x")
plt.ylabel("y")
plt.title("Schools coordinates")

In [ ]:
import nutpie

In [ ]:
with pm.Model() as gp_visual:
    ls = pm.Gamma("ls", mu=5, sigma=1)
    cov = pm.gp.cov.ExpQuad(2, ls=ls)
    gp = pm.gp.Latent(cov_func=cov)
    theta = gp.prior("theta_grid", grid)
    trace = pm.sample_prior_predictive(4, var_names=["theta_grid"])

In [ ]:
trace.prior.theta_grid.values.squeeze()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, draw in enumerate(trace.prior.theta_grid.values.squeeze()):
    gridz = draw.reshape(20, 20)
    axes[i].contourf(*gridxy, gridz)

In [ ]:
with pm.Model(coords=dict(J=range(J))) as schools_gp:
    mu = pm.Normal("mu", mu=0, sigma=5)
    tau = pm.HalfCauchy("tau", beta=5)
    
    # до
    # theta_tilde = pm.Normal("theta_t", mu=0, sigma=1, dims="J")
    # после
    ls = pm.Gamma("ls", mu=5, sigma=1)
    cov = pm.gp.cov.ExpQuad(2, ls=ls)
    gp = pm.gp.Latent(cov_func=cov)
    theta_tilde = gp.prior("theta_t", school_coords, dims="J")
    
    theta = pm.Deterministic("theta", mu + tau * theta_tilde, dims="J")
    obs = pm.Normal("obs", mu=theta, sigma=sigma, observed=y)
    trace2 = pm.sample(target_accept=0.90)
    
    # Экстраполяция
    theta_tilde_grid = gp.conditional("theta_t_grid", grid)
    theta_grid = pm.Deterministic("theta_grid", mu + tau * theta_tilde_grid)
    # в ArviZ 1.x у DataTree нет .extend, поэтому добавляем группу напрямую
    ppc2 = pm.sample_posterior_predictive(trace2.sel(draw=np.s_[::100]), var_names=["theta_grid"])
    trace2["posterior_predictive"] = ppc2["posterior_predictive"]

In [ ]:
red = {"color": "red"}
ax = az.plot_forest(trace2, var_names=["theta"], visuals=dict(trunk=red, twig=red, point_estimate=red))

In [ ]:
az.plot_forest(trace1, var_names=["theta"])

In [ ]:
gridxy.shape

In [ ]:
gridz = trace2.posterior_predictive["theta_grid"].mean(["chain", "draw"]).values.reshape(20, 20)

In [ ]:
plt.contourf(*gridxy, gridz)
plt.scatter(*school_coords.T, c=y, edgecolors="w")
[plt.text(x, y+1, str(i)) for i, (x, y) in enumerate(school_coords)];
plt.colorbar();
plt.title("GP posterior")
plt.xlabel("x")
plt.ylabel("y")

In [ ]:
X = np.random.randn(100, 2)
Y = np.sin(X[:, 0]*5)*4 + (X[:, 1]) ** 2 + np.random.randn(100) *0.01